# Figure 1 - GNN-Pearson Feature Graph, labeled (v2, doc-matched)

Generates the model feature graph: 40-node economic network with the doc's **italic
subtitle note**, **horizontal node labels**, and two compact, **borderless, equal-size,
symmetric** elements at the bottom - the **mini feature-vector graph** (bottom-left) and
the **legend** (bottom-right).

- **GDP hub** (gold, center) connects to all 39 predictors.
- **Inner ring** (blue): 20 FRED macro indicators; **Outer ring** (green): 19 MRTS retail series.
- **Edges** at |rho| >= 0.7 - red (FRED-FRED), green (MRTS-MRTS), purple (cross-ring); gold spokes = GDP hub.
- **Inset:** per-node feature vector - 4 channels x 6-month window = 24 dims/node.

Data load matches `V72_Final.ipynb`. All display labels live in the `LBL` dict (cell 1).
The figure is saved as `fig_1_feature_graph_labeled.png` in this same folder.

In [ ]:
import pandas as pd, numpy as np, pathlib, itertools
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

DATA_DIR = pathlib.Path("./data")
FRED_DIR = DATA_DIR / "input/new/ref_from_ST_FRED"
MRTS_DIR = DATA_DIR / "input/new/raw_from_MRTS"
GDP_CSV  = FRED_DIR / "Quarterly_GDP.csv"
OUTPUT_DIR = DATA_DIR / "output/new/GNN_corr/quarterly_walkfwd_v72_final"

FRED = ["F_PCE_Pers","F_PCES_Serv","F_AHETPI_Avg","F_CPIAUCSL_Cons","F_DSPI_Disp",
        "F_TOTALSL_Tota","F_Real_Pers","F_M2SL_M2","F_PCEDG_Dura","F_BUSLOANS_Comm",
        "F_Real_Disp","F_PPIACO_PPI","F_PAYEMS_Nonf","F_DGORDER_Dura","F_NASDAQCOM_NASD",
        "F_NEWORDER_Core","F_INDPRO_Indu","F_MANEMP_ISM","F_GS10_10Ye","F_WTISPLC_WTI"]
MRTS = ["MRTS_44Y72","MRTS_44W72","MRTS_44X72","MRTS_44Z72","MRTS_44000","MRTS_446",
        "MRTS_445","MRTS_722","MRTS_4451","MRTS_452","MRTS_454","MRTS_441","MRTS_444",
        "MRTS_44112","MRTS_448","MRTS_453","MRTS_447","MRTS_442","MRTS_4522"]
LBL = {"F_PCE_Pers":"PCE","F_PCES_Serv":"PCE Services","F_AHETPI_Avg":"Hourly Earnings",
       "F_CPIAUCSL_Cons":"CPI","F_DSPI_Disp":"Disp. Income","F_TOTALSL_Tota":"Consumer Credit",
       "F_Real_Pers":"Real PCE","F_M2SL_M2":"M2 Money","F_PCEDG_Dura":"PCE Durables",
       "F_BUSLOANS_Comm":"Bus. Loans","F_Real_Disp":"Real DPI","F_PPIACO_PPI":"PPI",
       "F_PAYEMS_Nonf":"Nonfarm Pay","F_DGORDER_Dura":"Durable Goods","F_NASDAQCOM_NASD":"NASDAQ",
       "F_NEWORDER_Core":"Capital Orders","F_INDPRO_Indu":"Ind. Prod.","F_MANEMP_ISM":"Mfg. Emp.",
       "F_GS10_10Ye":"10Y Treasury","F_WTISPLC_WTI":"WTI Oil","MRTS_44000":"Total Retail",
       "MRTS_441":"Auto & Parts","MRTS_442":"Furniture","MRTS_444":"Bldg. Materials",
       "MRTS_445":"Food & Bev.","MRTS_4451":"Grocery","MRTS_446":"Health Care","MRTS_447":"Gas Stations",
       "MRTS_448":"Clothing","MRTS_452":"Gen. Merch.","MRTS_4522":"Dept. Stores","MRTS_453":"Misc. Retail",
       "MRTS_454":"E-Commerce","MRTS_722":"Food Services","MRTS_44112":"Auto Dealers",
       "MRTS_44X72":"Retail+Food Svc","MRTS_44W72":"Retail ex Auto","MRTS_44Y72":"Retail ex Gas",
       "MRTS_44Z72":"Total ex Gas"}

def load_fred():
    d=[]
    for f in sorted(FRED_DIR.glob("Monthly_*.csv")):
        df=pd.read_csv(f,parse_dates=["observation_date"]); vc=[c for c in df.columns if c!="observation_date"][0]
        df.rename(columns={"observation_date":"date"},inplace=True); df[vc]=pd.to_numeric(df[vc],errors="coerce")
        df=df.dropna(subset=[vc]).set_index("date")[[vc]]; p=f.stem.replace("Monthly_","").split("_")
        df.columns=[f"F_{p[0]}_{p[1][:4]}" if len(p)>=2 else f"F_{p[0]}"]; d.append(df)
    return pd.concat(d,axis=1).sort_index().resample("MS").last()

def load_mrts():
    d=[]
    for f in sorted(MRTS_DIR.glob("*.xlsx")):
        try:
            n=f.name.split("_")[0]; raw=pd.read_excel(f,header=None,skiprows=7)
            raw.columns=["Period","Value"]+[f"x{i}" for i in range(len(raw.columns)-2)]
            raw["Value"]=pd.to_numeric(raw["Value"],errors="coerce")
            raw["date"]=pd.to_datetime(raw["Period"].astype(str),format="%b-%Y",errors="coerce")
            raw=raw.dropna(subset=["date","Value"]).set_index("date")[["Value"]]; raw.columns=[f"MRTS_{n}"]; d.append(raw)
        except Exception: pass
    return pd.concat(d,axis=1).sort_index().resample("MS").last()
print("Loaders, 39-feature set, and label map defined.")


In [ ]:
fred_df=load_fred(); mrts_df=load_mrts()
gdp=pd.read_csv(GDP_CSV,parse_dates=["observation_date"]).rename(columns={"observation_date":"date"})
gdp["GDP"]=pd.to_numeric(gdp["GDP"],errors="coerce"); gdp=gdp.dropna(subset=["GDP"]).set_index("date")[["GDP"]]
idx=pd.date_range(min(fred_df.index.min(),mrts_df.index.min()),max(fred_df.index.max(),mrts_df.index.max()),freq="MS")
merged=pd.concat([fred_df,mrts_df,gdp.shift(1).reindex(idx).ffill()],axis=1).sort_index()
fred=[c for c in FRED if c in merged.columns]; mrts=[c for c in MRTS if c in merged.columns]
C=merged[fred+mrts+["GDP"]].loc["2006-01-01":"2025-12-01"].dropna().corr(); TAU=0.7
nedges=sum(1 for a,b in itertools.combinations(fred+mrts,2) if abs(C.loc[a,b])>=TAU)
print(f"Nodes: {len(fred)+len(mrts)+1} ({len(fred)} FRED + {len(mrts)} MRTS + GDP) | edges |rho|>={TAU}: {nedges} ({nedges/741*100:.0f}%)")


In [ ]:
RF, RM = 1.6, 3.15
pos={"GDP":(0,0)}; ang={}
for i,n in enumerate(fred): a=2*np.pi*i/len(fred)+np.pi/2;      ang[n]=a; pos[n]=(RF*np.cos(a),RF*np.sin(a))
for i,n in enumerate(mrts): a=2*np.pi*i/len(mrts)+np.pi/2+0.08; ang[n]=a; pos[n]=(RM*np.cos(a),RM*np.sin(a))

fig,ax=plt.subplots(figsize=(16,16))
for n in fred+mrts: ax.plot([0,pos[n][0]],[0,pos[n][1]],color="#C9A227",alpha=0.10,lw=0.6,zorder=1)
for a,b in itertools.combinations(fred+mrts,2):
    if abs(C.loc[a,b])>=TAU:
        af,bf=a in fred,b in fred; col="#d62728" if(af and bf)else("#2ca02c" if(not af and not bf)else "#9467bd")
        ax.plot([pos[a][0],pos[b][0]],[pos[a][1],pos[b][1]],color=col,alpha=0.16,lw=0.5,zorder=2)
ax.scatter([pos[n][0] for n in fred],[pos[n][1] for n in fred],s=430,c="#2f6db5",edgecolors="white",lw=1.3,zorder=4)
ax.scatter([pos[n][0] for n in mrts],[pos[n][1] for n in mrts],s=430,c="#2f9e44",edgecolors="white",lw=1.3,zorder=4)
ax.scatter([0],[0],s=2600,c="#B8860B",edgecolors="black",lw=1.5,zorder=5)
ax.text(0,0,"GDP\nHub",ha="center",va="center",fontsize=12,fontweight="bold",color="white",zorder=6)

def hlabel(n,rn,color):
    a=ang[n]; off=0.30; lx,ly=(rn+off)*np.cos(a),(rn+off)*np.sin(a); deg=np.degrees(a)%360
    if deg<55 or deg>305: ha,va='left','center'
    elif 55<=deg<=125: ha,va='center','bottom'
    elif 125<deg<235: ha,va='right','center'
    else: ha,va='center','top'
    ax.text(lx,ly,LBL[n],ha=ha,va=va,fontsize=8.5,color=color,zorder=6)
for n in fred: hlabel(n,RF,"#16385f")
for n in mrts: hlabel(n,RM,"#176317")
ax.set_xlim(-4.5,4.5); ax.set_ylim(-4.5,3.85); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Figure 1.  GNN-Pearson Feature Graph: 40-Node Economic Network",fontsize=15,fontweight="bold",pad=24)
ax.text(0.5,1.012,"GDP hub (gold) connects unconditionally to all 39 predictor nodes    |    Edges shown where |ρ| ≥ 0.7 on training data    |    Each node carries $x_i(t)\in\mathbb{R}^{24}$  (4 channels × 6-month window)",transform=ax.transAxes,ha="center",va="center",fontsize=10.5,style="italic",color="#555555")

# ---- two equal-size, symmetric, borderless boxes (smaller) ----
BW, BH, BY = 0.17, 0.17, 0.05
def style_box(a):                 # borderless: blends into the figure
    a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values(): s.set_visible(False)

# mini feature-vector star-graph (bottom-LEFT)
iax=fig.add_axes([0.05, BY, BW, BH]); style_box(iax)
iax.set_xlim(-1.35,1.35); iax.set_ylim(-1.45,1.30)
iax.add_patch(plt.Circle((0,0),0.24,color='#3a3a3a',zorder=3)); iax.text(0,0,"Node",color='w',ha='center',va='center',fontsize=6,fontweight='bold',zorder=4)
for name,col,(cx,cy) in [("Levels","#2f6db5",(-0.78,0.58)),("Velocity","#2f9e44",(0.78,0.58)),("Volatility","#9467bd",(-0.78,-0.62)),("Accel.","#d62728",(0.78,-0.62))]:
    iax.plot([0,cx],[0,cy],color='gray',lw=0.9,zorder=1)
    for k in range(6):
        t=2*np.pi*k/6; iax.scatter(cx+0.30*np.cos(t),cy+0.30*np.sin(t),s=7,color=col,alpha=0.8,zorder=2)
    iax.add_patch(plt.Circle((cx,cy),0.17,color=col,zorder=3)); iax.text(cx,cy,name,color='w',ha='center',va='center',fontsize=4.6,fontweight='bold',zorder=4)
iax.text(0,-1.34,"4 ch x 6 mo = 24 dims/node",ha='center',fontsize=5.2)
iax.set_title("Per-Node Feature Vector  $X_i(t)$",fontsize=8,fontweight='bold')

# legend (bottom-RIGHT) — mirror of the inset
lax=fig.add_axes([1-0.05-BW, BY, BW, BH]); style_box(lax)
leg=[Line2D([0],[0],marker='o',color='w',markerfacecolor='#2f6db5',markersize=9,label='FRED Macro node (20)'),
     Line2D([0],[0],marker='o',color='w',markerfacecolor='#2f9e44',markersize=9,label='MRTS Retail node (19)'),
     Line2D([0],[0],marker='o',color='w',markerfacecolor='#B8860B',markeredgecolor='k',markersize=11,label='GDP Hub node (1)'),
     Line2D([0],[0],color='#d62728',lw=1.8,label='FRED-FRED correlation edge'),
     Line2D([0],[0],color='#2ca02c',lw=1.8,label='MRTS-MRTS correlation edge'),
     Line2D([0],[0],color='#9467bd',lw=1.8,label='FRED-MRTS cross-ring edge'),
     Line2D([0],[0],color='#C9A227',lw=1.8,ls='--',alpha=0.8,label='GDP hub spoke (all nodes)')]
lg=lax.legend(handles=leg,loc='center',frameon=False,fontsize=7.5,handlelength=1.3,labelspacing=0.6,borderaxespad=0.1,title="Legend",title_fontsize=8.5)
lg.get_title().set_fontweight('bold'); lg._legend_box.sep=2

FIG_DIR = pathlib.Path("./figures")
fig.savefig(FIG_DIR/"fig_1_feature_graph_labeled.png",dpi=150,bbox_inches="tight")
print("Saved:", FIG_DIR/"fig_1_feature_graph_labeled.png")


## Notes
- 40 nodes (20 FRED + 19 MRTS + GDP); edges at |rho| >= 0.7 on full 2006-2025 sample (~564 edges, 76%); per-fold ~650 (88%).
- Title carries the doc's italic subtitle note, snug above the top ring (tightened top margin closes the gap).
- The bottom-left feature-vector graph and bottom-right legend are borderless, equal-size, and symmetric (`BW, BH, BY` in the draw cell).
- The four `44*72` MRTS series are Census special aggregates - edit `LBL` if your codebook differs.
- Saved as `fig_1_feature_graph_labeled.png` (dpi 150) alongside this notebook.